# Data Cleaning

In [1]:
import pandas as pd
from pathlib import Path
import geopandas as gpd 

In [2]:
# Set project root directory (one level above the notebook folder)
BASE_DIR = Path().resolve().parent

# Read the data file
raw_file_path = BASE_DIR / "data/raw/stzh.zwn_meldungen_p.csv"
reports = pd.read_csv(raw_file_path)

# Date transformation
reports["requested_datetime"] = pd.to_datetime(reports["requested_datetime"])

# Data from the last 10 years
ten_years_ago = pd.Timestamp.now() - pd.DateOffset(years=10)
reports = reports[reports["requested_datetime"] > ten_years_ago]

# Date column
reports["date"] = reports["requested_datetime"].dt.date

# Extract useful columns
print(reports.columns)
reports = reports[["date","service_code", "e", "n", "geometry"]]

display(reports.sort_values(by="date", ascending=False).head())
display(reports.sort_values(by="date", ascending=True).head())

# Save the cleand .csv in the "processed" data folder
processed_file_path = BASE_DIR / "data/processed/reports_cleaned.csv"
reports.to_csv(processed_file_path, index=False)

Index(['objectid', 'service_request_id', 'requested_datetime',
       'agency_sent_datetime', 'updated_datetime', 'e', 'n', 'service_code',
       'service_name', 'status', 'userid', 'title', 'detail', 'media_url',
       'interface_used', 'service_notice', 'description', 'url', 'geometry',
       'date'],
      dtype='object')


,date,service_code,e,n,geometry
72186,2026-04-16,Abfall/Sammelstelle,2680495,1248455,POINT (2680495 1248455)
72182,2026-04-16,Allgemein,2681516,1246219,POINT (2681516 1246219)
72177,2026-04-16,Abfall/Sammelstelle,2685747,1246246,POINT (2685747 1246246)
72178,2026-04-16,Abfall/Sammelstelle,2682307,1249366,POINT (2682307 1249366)
72179,2026-04-16,Abfall/Sammelstelle,2679762,1251038,POINT (2679762 1251038)


,date,service_code,e,n,geometry
7486,2016-04-18,Strasse/Trottoir/Platz,2681456,1251793,POINT (2681456 1251793)
7488,2016-04-18,Abfall/Sammelstelle,2681880,1248999,POINT (2681880 1248999)
7489,2016-04-18,Signalisation/Lichtsignal,2687917,1245619,POINT (2687917 1245619)
7485,2016-04-18,Grünflächen/Spielplätze,2682712,1247955,POINT (2682712 1247955)
7487,2016-04-18,Signalisation/Lichtsignal,2683518,1246830,POINT (2683518 1246830)


In [13]:
# Lakes layer auf Stadt Zürich zuschneiden
# Read the data file
zh_quartiere_file_path = BASE_DIR / "data/raw/seen_stadtZH.gpkg"
zh_quariere = gpd.read_file(zh_quartiere_file_path)

lakes_file_path = BASE_DIR / "data/raw/AV_Gewasser_-OGD.gpkg"
lakes = gpd.read_file(lakes_file_path)
lakes = lakes.to_crs(epsg=2056)

perimeter = zh_quariere.union_all()

lakes_zh = gpd.clip(lakes, perimeter)

output_path = BASE_DIR / "data/processed/lakes_zh.gpkg"
lakes_zh.to_file(output_path, driver="GPKG")

